# Population measures and flows

**Question.** Where do people in a population live and gather, when are places busy, and how many people move between places?

**You will get** counts of homes and visits per place, activity by day, a measure of how far people spread over time, the diversity of visitors per place, origin-destination flows and their per-area summaries, a spatial flow table with a grid, a comparison of two flow tables, and a time-binned occupancy and co-visitation table.

**Prerequisites.** [Measure individual mobility](measure-individual-mobility.md).

**Data.** [Brightkite](https://snap.stanford.edu/data/loc-brightkite.html) check-ins (Stanford SNAP, 2008 to 2010): the first 300,000 rows (about 4 MB of the 60 MB file) with `fastmob.data.load_brightkite`; users with fewer than 100 check-ins are dropped, which leaves about 306 users. Check-ins are voluntary and sparse. The grid used for zones is synthetic. The notebook stops if the download fails.

In [2]:
get_ipython().run_line_magic("pip", "install -q 'fastmob>=0.3.4' pandas matplotlib geopandas shapely")

In [4]:
import tempfile
from pathlib import Path

cache_status = "cached (files from an earlier run were reused)" if (Path(tempfile.gettempdir()) / "fastmob_data" / "brightkite").exists() else "cold (nothing cached before this run)"

## Load and clean

The raw file contains placeholder and impossible coordinates. We drop check-ins at exactly latitude 0 and longitude 0 (a placeholder for "unknown") and the few rows with a latitude whose absolute value is 90 degrees or more, which is not a valid position. Always check coordinate ranges before a population analysis.

In [5]:
import fastmob
import numpy as np
import pandas as pd
from fastmob.data import load_brightkite
from fastmob.measures import collective

raw = load_brightkite(max_rows=300_000).to_pandas()

bad = ((raw["lat"] == 0) & (raw["lng"] == 0)) | (raw["lat"].abs() >= 90)
print(f"dropping {bad.sum()} check-ins with placeholder or invalid coordinates")
checkins = raw[~bad]
counts = checkins.groupby("uid").size()
checkins = checkins[checkins["uid"].isin(counts[counts >= 100].index)].sort_values(["uid", "datetime"]).reset_index(drop=True)
print(len(checkins), "check-ins by", checkins["uid"].nunique(), "users")

dropping 8327 check-ins with placeholder or invalid coordinates
283543 check-ins by 306 users


## Homes and visits per place

`homes_per_location` counts the users whose inferred home is at each coordinate (home is the most visited place at night, in UTC here). `visits_per_location` counts visits per place id; we use H3 cells of about 0.7 km (resolution 8) as place ids. `visits_per_time_unit` counts check-ins per day.

In [6]:
homes = collective.homes_per_location(checkins)
cells = fastmob.latlng_to_h3(checkins, resolution=8).rename(columns={"h3_cell": "location_id"})
visits = collective.visits_per_location(cells, location_id_col="location_id", uid_col="uid", datetime_col="datetime")
per_day = collective.visits_per_time_unit(checkins, freq="1D")

print("most shared homes:")
print(homes.nlargest(3, "n_homes").to_string(index=False))
print("busiest places:")
print(visits.nlargest(3, "n_visits").to_string(index=False))
per_day.nlargest(3, "n_visits")

most shared homes:
      lat         lng  n_homes
39.739154 -104.984703       13
37.774929 -122.419415        8
37.580304 -122.343679        4
busiest places:
       location_id  n_visits
613167731774062591      5174
613196574496915455      3583
613166961024565247      2684


,datetime,n_visits
860,2009-03-17,891
112,2009-09-26,622
858,2009-03-13,605


The most shared home coordinate is in central Denver (13 users), then central San Francisco (8). The busiest H3 cell has 5,174 check-ins. Check-in activity peaks on single days: 891 on 17 March 2009, the busiest day, which suggests an event or a surge in sign-ups rather than a daily rhythm.

## How far do people spread over time?

`mean_square_displacement` is the average squared distance (km²) people have moved from their starting point after the chosen time (here 1 day), averaged over users. A value that grows linearly with the time lag is the signature of ordinary diffusion; faster or slower growth indicates other regimes.

In [7]:
msd = {d: collective.mean_square_displacement(checkins, days=d, hours=0) for d in (1, 7)}
msd

{1: 915757.6763045556, 7: 658942.7669518194}

After one day the mean squared displacement is about 916,000 km² (a root-mean-square distance near 960 km), and after seven days about 659,000 km²: it does not grow with the time lag. Check-ins are sparse and a few users travel between continents, so the average rests on a small number of long moves. Treat it as a description of this sample, not as evidence of a diffusion regime.

## How many different people use a place?

`random_location_entropy` is the entropy of the number of distinct visitors of each place (log2 of that number), and `uncorrelated_location_entropy` weights visitors by how often they visit. Both are 0 for places visited by one person and grow with the diversity of visitors.

In [8]:
random_h = collective.random_location_entropy(checkins)
uncorrelated_h = collective.uncorrelated_location_entropy(checkins)

print(len(random_h), "distinct coordinates;", f"{(random_h['random_entropy'] == 0).mean():.0%} have a single visitor")
random_h.nlargest(3, "random_entropy").merge(uncorrelated_h, on=["lat", "lng"])

43520 distinct coordinates; 90% have a single visitor


,lat,lng,random_entropy,uncorrelated_entropy
0,39.739154,-104.984703,6.357552,4.519015
1,37.774929,-122.419415,6.321928,4.179255
2,37.615223,-122.389979,5.882643,5.190807


There are 43,520 distinct coordinates and 90% of them have a single visitor. The top coordinate (central Denver) has a random entropy of 6.4 bits, which corresponds to about 82 distinct visitors, and an uncorrelated entropy of only 4.5 bits: a few visitors account for most of the check-ins there.

## From check-ins to flows

`trajectory_to_trips` turns consecutive check-ins into trips between H3 cells (resolution 7, about 2.5 km), and `trajectory_to_od` counts them as an origin-destination table. `od_matrix` aggregates a trips table by origin and destination area; `od_metrics_per_area` summarises each area: trips inside it (`MoveInside`), arriving (`InComing`), leaving (`OutGoing`) and in total.

In [9]:
trips = fastmob.trajectory_to_trips(checkins, resolution=7)
od = fastmob.trajectory_to_od(checkins, resolution=7)
areas = trips.to_pandas().rename(columns={"origin_location_id": "origin_area", "destination_location_id": "destination_area"})
matrix = collective.od_matrix(areas)
per_area = collective.od_metrics_per_area(od, origin_col="origin", destination_col="destination")

print(len(trips), "trips;", len(od), "distinct origin-destination pairs")
per_area.nlargest(3, "Total")

111610 trips; 45606 distinct origin-destination pairs


,area_code,MoveInside,InComing,OutGoing,Total
3100,608664132161372159,0,3438,3443,6881
3104,608664132228481023,0,2003,2004,4007
2617,608663361399291903,0,1858,1863,3721


The check-ins make 111,610 trips and 45,606 distinct origin-destination pairs at resolution 7. `MoveInside` is 0 for every area because trips within one cell are dropped by default. The busiest cell has 6,881 trips in total (3,438 arriving, 3,443 leaving), and arrivals and departures are close for the busiest cells, as expected when people return to where they started.

## Flows between zones

For zone-based analysis, a `TrajDataFrame` converts to a `FlowDataFrame` given a tessellation (polygons with a `tile_id`). We use a synthetic 10 by 10 grid over the contiguous United States, and keep only check-ins inside it. `get_flow` reads one origin-destination flow, `get_geometry` returns a tile's polygon, `to_matrix` the full origin-destination matrix.

In [10]:
import geopandas as gpd
from shapely.geometry import box

n, (lat0, lat1, lng0, lng1) = 10, (25, 50, -125, -65)
grid = gpd.GeoDataFrame(
    {"tile_id": [str(k) for k in range(n * n)]},
    geometry=[box(lng0 + j * (lng1 - lng0) / n, lat0 + i * (lat1 - lat0) / n, lng0 + (j + 1) * (lng1 - lng0) / n, lat0 + (i + 1) * (lat1 - lat0) / n) for i in range(n) for j in range(n)],
    crs="EPSG:4326",
)
inside = checkins[checkins["lat"].between(lat0, lat1) & checkins["lng"].between(lng0, lng1)]

flows = fastmob.TrajDataFrame(inside).to_flowdataframe(grid, self_loops=False)
table = flows.to_pandas()
top = table.nlargest(1, "flow").iloc[0]
matrix_z = flows.to_matrix()
one_flow = flows.get_flow(top["origin"], top["destination"])
tile = flows.get_geometry(top["origin"])

print(len(inside), "check-ins inside the grid;", len(table), "zone pairs with flow; matrix", matrix_z.shape)
print("largest flow:", top["origin"], "->", top["destination"], "=", one_flow, "trips; origin tile polygon:", tile.wkt)

239549 check-ins inside the grid; 1117 zone pairs with flow; matrix (100, 100)
largest flow: 40 -> 50 = 1692 trips; origin tile polygon: POLYGON ((-119 35, -119 37.5, -125 37.5, -125 35, -119 35))


The grid has 1,117 zone pairs with flow; the largest, 1,692 trips, joins two neighbouring tiles on the California coast. `get_geometry` returns the tile's polygon as shapely geometry and `to_matrix` the full 100 by 100 matrix. The grid is synthetic and its tiles are several hundred kilometres wide.

## Compare two flow tables

Split the users into two random halves and compare their flow tables. `common_part_of_commuters` (CPC) is the share of trips that both tables send along the same links (1 means identical, 0 completely different) and `common_part_of_links` (CPL) the share of links they share. The same two metrics exist for `Trips` tables. `settings_from` copies the tessellation and settings from one flow table to another.

In [11]:
users = inside["uid"].unique()
half = set(np.random.default_rng(0).permutation(users)[: len(users) // 2])

flows_a = fastmob.TrajDataFrame(inside[inside["uid"].isin(half)]).to_flowdataframe(grid, self_loops=False)
flows_b = fastmob.TrajDataFrame(inside[~inside["uid"].isin(half)]).to_flowdataframe(grid, self_loops=False)
cpc, cpl = flows_a.common_part_of_commuters(flows_b), flows_a.common_part_of_links(flows_b)
flows_a.settings_from(flows_b)
trips_a = fastmob.trajectory_to_trips(inside[inside["uid"].isin(half)], resolution=7)
trips_b = fastmob.trajectory_to_trips(inside[~inside["uid"].isin(half)], resolution=7)
trips_cpc, trips_cpl = trips_a.common_part_of_commuters(trips_b), trips_a.common_part_of_links(trips_b)
as_flow = trips_a.to_flow_dataframe()

print(f"zones:  CPC {cpc:.2f}, CPL {cpl:.2f}")
print(f"H3 trips: CPC {trips_cpc:.2f}, CPL {trips_cpl:.2f}; {len(as_flow)} H3 flow pairs for half A")

zones:  CPC 0.65, CPL 0.62
H3 trips: CPC 0.15, CPL 0.11; 20720 H3 flow pairs for half A


The two halves agree well at zone level (CPC 0.65: about two thirds of the trips follow links present in both) and poorly at the 2.5 km cell level (CPC 0.15, CPL 0.11). Agreement between two flow tables depends strongly on the spatial resolution, so always report the resolution next to CPC.

## When and where is each place busy?

To study occupancy we need *stays*. Check-ins have no duration, so we **assume** each check-in is a 30-minute stay (an explicit modelling choice, not data). `generate_global_locations` assigns the stays to a shared H3 grid (resolution 8). `build_stvd` then aggregates them into the average number of people at each place in each 10-minute time bin, `interest_network` (also available as `collective.interest_network`) lists pairs of places visited by the same people, and `mean_area_volume` does the same occupancy average for any visit table.

In [12]:
stays = fastmob.Staypoints(
    pd.DataFrame({"uid": checkins["uid"], "lat": checkins["lat"], "lng": checkins["lng"],
                  "started_at": checkins["datetime"], "finished_at": checkins["datetime"] + pd.Timedelta(minutes=30)})
)

places, stays_at = stays.generate_global_locations(h3_resolution=8)
stvd = stays_at.build_stvd(places)
shared = stays_at.interest_network(places)
busy = collective.mean_area_volume(
    stays_at.to_pandas().rename(columns={"uid": "user_id"}),
    area_col="location_id", user_id_col="user_id", start_col="started_at", end_col="finished_at",
)
stvd_f = collective.build_stvd(stays_at, places)
shared_f = collective.interest_network(stays_at, places)

same = stvd.sort_values(list(stvd.columns)).reset_index(drop=True).equals(stvd_f.sort_values(list(stvd_f.columns)).reset_index(drop=True))
print(len(places), "places;", len(stvd), "place-time cells;", len(shared), "linked place pairs; method and function agree:", same, len(shared_f) == len(shared))
shared.nlargest(3, "n_people")

18659 places; 291182 place-time cells; 2461194 linked place pairs; method and function agree: True True


,location_id_a,location_id_b,n_people
182,613196570331971583,613196570376011775,71
9079,613196574729699327,613196570376011775,65
19781,613167731774062591,613167731786645503,63


Assigning the stays to a shared grid gives 18,659 places and 291,182 place-time cells, and 2.46 million pairs of places visited by the same people; the most strongly linked pair is shared by 71 people. Only the head of each table is shown. `build_stvd` called as a method and as a function give the same table.

## Mobile-phone records

Call detail records (CDR) give the cell tower a person was connected to at the time of each event, not their position. `cdr_to_visitation_df` collapses consecutive records at the same tower into a stay and estimates when each stay ended from the travel time to the next tower (`avg_speed_kmh` and `circuity` set the assumed speed and route winding); `cdr_to_trips_df` derives the trips between consecutive stays. **The records below are synthetic**: six events of one invented person at two invented towers.

In [13]:
from fastmob.preprocessing import cdr_to_trips_df, cdr_to_visitation_df

cdr = pd.DataFrame({
    "user_id": ["u1"] * 6,
    "timestamp": pd.to_datetime(["2020-01-01 08:00", "2020-01-01 08:15", "2020-01-01 09:00", "2020-01-01 12:00", "2020-01-01 17:00", "2020-01-01 22:00"]),
    "venueId": ["tower_home", "tower_home", "tower_work", "tower_work", "tower_work", "tower_home"],
    "lat": [41.890, 41.890, 41.911, 41.911, 41.911, 41.890],
    "long": [12.492, 12.492, 12.482, 12.482, 12.482, 12.492],
})
cdr_visits = cdr_to_visitation_df(cdr, avg_speed_kmh=30, circuity=1.2)
cdr_trips = cdr_to_trips_df(cdr_visits, avg_speed_kmh=30, circuity=1.2)

print(cdr_visits[["user_id", "area", "start_timestamp", "end_timestamp", "duration_minutes"]].round(1).to_string(index=False))
print(cdr_trips[["user_id", "trip_number", "origin_area", "destination_area", "duration_minutes"]].round(1).to_string(index=False))

user_id       area     start_timestamp              end_timestamp  duration_minutes
     u1 tower_home 2020-01-01 08:00:00 2020-01-01 08:54:03.250349              54.1
     u1 tower_work 2020-01-01 09:00:00 2020-01-01 21:54:03.250349             774.1
     u1 tower_home 2020-01-01 22:00:00                        NaT               NaN
user_id  trip_number origin_area destination_area  duration_minutes
     u1            1  tower_home       tower_work               5.9
     u1            2  tower_work       tower_home               5.9


The estimated end of the home stay is 08:54, about six minutes before the next record at 09:00 (the estimated 5.9-minute trip at 30 km/h with a circuity of 1.2); the work stay is estimated to end at 21:54 for the same reason; the last stay has no end (`NaT`) because no record follows it. The two trips between the towers last 5.9 minutes each. These end times are model estimates built from the assumed speed, not observations, and a sparse CDR can hide whole trips.

## Interpretation and limits

- Check-in data are not mobility data. People check in where and when they choose, so homes, entropies and flows describe check-in behaviour. The first 300,000 rows are not a random sample of users.
- Place ids come from a grid. Visit counts, entropy and flows all change with the grid size; state it, and test a second size.
- The occupancy tables assume each check-in lasts 30 minutes. Their absolute values follow that assumption.
- Flows join consecutive check-ins of the same user, regardless of the time between them, so a trip may cover days.
- CPC and CPL compare two random halves of the same sample; a real comparison, such as model against data, uses the evaluation metrics ([Fit and compare distributions](fit-and-evaluate.md)).

## Next steps

- Evaluate a flow model against observed flows: [Fit and compare distributions](fit-and-evaluate.md), [Generate mobility with models](generate-mobility.md).
- Build contact networks from co-presence: [Contact networks from stays](contact-networks.md).

## Related API

[`homes_per_location`](../reference/measures/collective.md#fastmob.measures.collective.homes_per_location.homes_per_location) ·
[`visits_per_location`](../reference/measures/collective.md#fastmob.measures.collective.visits_per_location.visits_per_location) ·
[`visits_per_time_unit`](../reference/measures/collective.md#fastmob.measures.collective.visits_per_time_unit.visits_per_time_unit) ·
[`mean_square_displacement`](../reference/measures/collective.md#fastmob.measures.collective.mean_square_displacement.mean_square_displacement) ·
[`random_location_entropy`](../reference/measures/collective.md#fastmob.measures.collective.random_location_entropy.random_location_entropy) ·
[`uncorrelated_location_entropy`](../reference/measures/collective.md#fastmob.measures.collective.uncorrelated_location_entropy.uncorrelated_location_entropy) ·
[`od_matrix`](../reference/measures/collective.md#fastmob.measures.collective.od_matrix) ·
[`od_metrics_per_area`](../reference/measures/collective.md#fastmob.measures.collective.od_metrics_per_area) ·
[`build_stvd`](../reference/measures/collective.md#fastmob.measures.collective.build_stvd) ·
[`mean_area_volume`](../reference/measures/collective.md#fastmob.measures.collective.mean_area_volume) ·
[`trajectory_to_trips`](../reference/preprocessing.md#fastmob.preprocessing.trajectory_to_trips) ·
[`trajectory_to_od`](../reference/preprocessing.md#fastmob.preprocessing.trajectory_to_od) ·
[`get_flow`](../reference/data-structures/flow-dataframe.md#fastmob.core.flow_dataframe.FlowDataFrame.get_flow) ·
[`get_geometry`](../reference/data-structures/flow-dataframe.md#fastmob.core.flow_dataframe.FlowDataFrame.get_geometry) ·
[`to_matrix`](../reference/data-structures/flow-dataframe.md#fastmob.core.flow_dataframe.FlowDataFrame.to_matrix) ·
[`settings_from`](../reference/data-structures/flow-dataframe.md#fastmob.core.flow_dataframe.FlowDataFrame.settings_from) ·
[`common_part_of_commuters`](../reference/data-structures/flow-dataframe.md#fastmob.core.flow_dataframe.FlowDataFrame.common_part_of_commuters) ·
[`common_part_of_links`](../reference/data-structures/flow-dataframe.md#fastmob.core.flow_dataframe.FlowDataFrame.common_part_of_links) ·
[`to_flow_dataframe`](../reference/data-structures/trips.md#fastmob.core.trips_dataframe.Trips.to_flow_dataframe) ·
[`common_part_of_commuters`](../reference/data-structures/trips.md#fastmob.core.trips_dataframe.Trips.common_part_of_commuters) ·
[`common_part_of_links`](../reference/data-structures/trips.md#fastmob.core.trips_dataframe.Trips.common_part_of_links) ·
[`build_stvd`](../reference/data-structures/staypoints.md#fastmob.core.staypoints_dataframe.Staypoints.build_stvd) ·
[`interest_network`](../reference/data-structures/staypoints.md#fastmob.core.staypoints_dataframe.Staypoints.interest_network)
